# Notebook 153: 文・文書の埋め込み (Sentence & Document Embeddings)

## From Words to Sentences: Capturing Meaning Beyond Individual Tokens

---

### このノートブックの位置づけ

**Embeddingsシリーズ** の第4章として、単語レベルの埋め込みから一歩進み、  
**文・文書レベルの埋め込み（Sentence & Document Embeddings）** を学びます。

難易度: ★★★☆☆ | 所要時間: 90-120分 | カテゴリ: Embeddings

### 学習目標

- [ ] 単語埋め込みから文埋め込みへの拡張が必要な理由を説明できる
- [ ] プーリング戦略（CLS/Mean/Max）の違いを実装し比較できる
- [ ] sentence-transformersライブラリを使って文の意味的類似度を計算できる
- [ ] 文埋め込みを用いた意味検索（Semantic Search）を実装できる
- [ ] 文埋め込みの品質を評価する方法を理解できる

### 前提知識

- ✅ Notebook 150（埋め込みの幾何学）
- ✅ Notebook 152（文脈付き埋め込み — BERTからの抽出）
- ✅ PyTorch基礎（Notebook 35-36）

---

## 目次

1. [環境セットアップ](#1-環境セットアップ)
2. [なぜ「文の埋め込み」が必要か？](#2-なぜ文の埋め込みが必要か)
3. [プーリング戦略の比較](#3-プーリング戦略の比較)
4. [Sentence-Transformers の使い方](#4-sentence-transformers-の使い方)
5. [意味検索（Semantic Search）の実装](#5-意味検索semantic-searchの実装)
6. [Sentence-BERTの学習の仕組み](#6-sentence-bertの学習の仕組み)
7. [文埋め込みの品質評価](#7-文埋め込みの品質評価)
8. [文書レベルへの拡張](#8-文書レベルへの拡張)
9. [まとめ・チートシート・自己評価クイズ](#9-まとめチートシート自己評価クイズ)

---

## 1. 環境セットアップ

In [ ]:
# ============================================================
# Section 1: 環境セットアップ
# 必要なライブラリの読み込みと初期設定
# ============================================================

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 再現性のためのシード設定
# ============================================================
np.random.seed(42)
torch.manual_seed(42)

# ============================================================
# 日本語フォント設定（matplotlib用）
# macOS: Hiragino Sans, フォールバック: Arial Unicode MS
# ============================================================
import japanize_matplotlib # 日本語化
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# ============================================================
# Seaborn スタイル設定
# ============================================================
sns.set_style('whitegrid')

# ============================================================
# デバイス設定（GPU が利用可能なら使用）
# ============================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用デバイス: {device}")

# ============================================================
# BERT モデルとトークナイザーの読み込み
# Notebook 152 で使用したものと同じモデル
# ============================================================
from transformers import BertTokenizer, BertModel

print("BERTモデルを読み込み中...")
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained(
    'bert-base-uncased',
    output_hidden_states=True
)
bert_model.eval()  # 推論モード（Dropout無効化）
print("BERTモデル読み込み完了")

# ============================================================
# Sentence-Transformers モデルの読み込み
# all-MiniLM-L6-v2: 軽量かつ高性能な文埋め込みモデル
# ============================================================
from sentence_transformers import SentenceTransformer, util

print("Sentence-Transformerモデルを読み込み中...")
st_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Sentence-Transformerモデル読み込み完了")
print(f"  モデル: all-MiniLM-L6-v2")
print(f"  出力次元: {st_model.get_sentence_embedding_dimension()}")

# ============================================================
# ヘルパー関数: コサイン類似度
# ============================================================
def cosine_sim(a, b):
    """2つのベクトル間のコサイン類似度を計算する。"""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print()
print("全ての準備が完了しました。")

---

## 2. なぜ「文の埋め込み」が必要か？

### 単語レベルから文レベルへ

Notebook 152 では、BERTを使って **単語レベル** の文脈付き埋め込みを学びました。  
しかし、多くの実用的なタスクでは **文全体** や **文書全体** を1つのベクトルで表現する必要があります。

### 文埋め込みが必要なユースケース

| ユースケース | 説明 | 必要な操作 |
|---|---|---|
| **意味検索（Semantic Search）** | クエリと文書の意味的類似度で検索 | 文ベクトル同士の比較 |
| **文書クラスタリング** | 類似文書を自動でグループ化 | 文書ベクトルのクラスタリング |
| **重複検出** | 同じ意味の文の重複を検出 | 文ベクトルの類似度閾値 |
| **質問応答** | 質問に最も関連する文を検索 | クエリと候補文の比較 |
| **文書分類** | 文書をカテゴリに分類 | 文書ベクトルの分類器 |

### BERTをそのまま文埋め込みに使う問題点

BERTは強力ですが、**文の類似度計算** にそのまま使うと大きな問題があります。

```
【Cross-Encoder アプローチ（BERTそのまま）】

  文A と 文B を同時にBERTに入力:
  
  [CLS] 文A [SEP] 文B [SEP]  →  BERT  →  類似度スコア
  
  問題: N個の文の全ペア比較 = N×(N-1)/2 回のBERT推論が必要！
  例: 10,000文 → 約5,000万回の推論 → 数日〜数週間

【Bi-Encoder アプローチ（Sentence-BERT）】

  文A → BERT → Pooling → ベクトルA ─┐
                                      ├→ コサイン類似度
  文B → BERT → Pooling → ベクトルB ─┘
  
  利点: 各文を1回だけエンコード（N回）
        → ベクトル同士の比較は超高速
  例: 10,000文 → 10,000回の推論 + 高速ベクトル比較 → 数秒
```

**Sentence-BERT（SBERT）** は、このBi-Encoderアプローチを
高品質に実現するために設計されたモデルです。

In [ ]:
# ============================================================
# Section 2: Cross-Encoder vs Bi-Encoder の計算コスト比較
# 文数に対する推論回数の違いをシミュレーション
# ============================================================

print("="*70)
print("Cross-Encoder vs Bi-Encoder: 計算コスト比較")
print("="*70)
print()

# 文数に対する推論回数を計算
# Cross-Encoder: 全ペア = N*(N-1)/2 回の推論
# Bi-Encoder: 各文を1回エンコード = N 回の推論
n_sentences_list = [10, 100, 1000, 10000, 100000]

print(f"{'文数 (N)':<15} {'Cross-Encoder':<25} {'Bi-Encoder':<20} {'速度比'}")
print("-" * 75)

for n in n_sentences_list:
    # Cross-Encoder: 全ペア比較
    cross_encoder_inferences = n * (n - 1) // 2
    # Bi-Encoder: 各文を1回だけエンコード
    bi_encoder_inferences = n
    # 速度比
    speedup = cross_encoder_inferences / bi_encoder_inferences
    
    print(f"{n:<15,} {cross_encoder_inferences:<25,} {bi_encoder_inferences:<20,} {speedup:,.0f}x")

print()
print("Cross-Encoderは文数の2乗に比例して計算量が増加します。")
print("Bi-Encoderは文数に線形で、大規模データセットでは圧倒的に高速です。")
print()
print("ただし、Cross-Encoderの方が精度は高い傾向があります。")
print("→ 実務では Bi-Encoder で候補を絞り、")
print("  Cross-Encoder でリランキングする2段階方式が一般的です。")

---

## 3. プーリング戦略の比較

BERTの出力から **文全体を表す1つのベクトル** を得るには、  
トークンレベルの埋め込みを「集約（プーリング）」する必要があります。

### 3つの代表的なプーリング戦略

| 戦略 | 方法 | 特徴 |
|---|---|---|
| **[CLS] トークン** | 最終層の[CLS]埋め込みをそのまま使用 | シンプル。分類タスク向けだが汎用性に欠ける |
| **Mean Pooling** | 全トークン埋め込みのattention_mask加重平均 | 安定して高性能。Sentence-BERTの標準 |
| **Max Pooling** | 各次元で最大値を取る | 顕著な特徴を強調。特定タスクで有効 |

In [ ]:
# ============================================================
# Section 3: プーリング戦略の実装
# BERTの出力から文ベクトルを得る3つの方法を実装
# ============================================================

def cls_pooling(model_output, inputs):
    """
    [CLS]トークンプーリング
    最終層の[CLS]トークン（位置0）の埋め込みを返す。
    
    BERTは[CLS]トークンを文全体の要約として学習しているが、
    文類似度タスクには最適化されていない。
    """
    # model_output.last_hidden_state: (batch, seq_len, hidden_dim)
    # [CLS] は常に位置 0
    return model_output.last_hidden_state[:, 0, :]


def mean_pooling(model_output, inputs):
    """
    Mean Pooling（attention_mask を考慮）
    パディングトークンを除外して平均を計算する。
    
    重要: attention_mask を無視すると、パディングトークンの
    ゼロベクトルが平均に含まれ、品質が低下する。
    """
    # token_embeddings: (batch, seq_len, hidden_dim)
    token_embeddings = model_output.last_hidden_state
    # attention_mask: (batch, seq_len) → (batch, seq_len, 1) に拡張
    attention_mask = inputs['attention_mask']
    mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    
    # マスクされたトークンの埋め込みをゼロにして合計
    sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
    # マスクの合計（有効トークン数）で割る
    sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
    
    return sum_embeddings / sum_mask


def max_pooling(model_output, inputs):
    """
    Max Pooling（attention_mask を考慮）
    各次元で最大値を取る。顕著な特徴を強調する。
    
    パディング位置は -1e9 に設定して max に影響しないようにする。
    """
    token_embeddings = model_output.last_hidden_state
    attention_mask = inputs['attention_mask']
    
    # パディング位置を非常に小さい値に設定
    mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    token_embeddings[mask_expanded == 0] = -1e9
    
    # 各次元の最大値を取る
    max_embeddings = torch.max(token_embeddings, dim=1)[0]
    return max_embeddings


def encode_with_bert(sentences, tokenizer, model, pooling_fn):
    """
    文リストをBERT + 指定プーリング戦略でエンコードする。
    
    Parameters:
    -----------
    sentences : list of str
        エンコードする文のリスト
    tokenizer : BertTokenizer
        BERTトークナイザー
    model : BertModel
        BERTモデル
    pooling_fn : callable
        プーリング関数（cls_pooling, mean_pooling, max_pooling）
    
    Returns:
    --------
    np.ndarray : (len(sentences), hidden_dim) の埋め込み行列
    """
    # トークナイズ（パディングあり、最大長は切り詰め）
    inputs = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )
    
    # 推論（勾配計算不要）
    with torch.no_grad():
        outputs = model(**inputs)
    
    # プーリング
    embeddings = pooling_fn(outputs, inputs)
    
    return embeddings.numpy()


print("プーリング関数の定義完了:")
print("  - cls_pooling():  [CLS]トークンを使用")
print("  - mean_pooling(): attention_mask加重平均")
print("  - max_pooling():  各次元の最大値")
print("  - encode_with_bert(): 文リストのバッチエンコード")

In [ ]:
# ============================================================
# Section 3 (続き): 3つのプーリング戦略を同じ文ペアで比較
# 意味的に類似/非類似な文ペアで類似度を計算
# ============================================================

# テスト用の文ペア
# 類似ペア (S) と非類似ペア (D) を交互に配置
test_pairs = [
    ("A cat is sitting on the mat",
     "A kitten rests on a soft rug",
     "類似"),
    ("A cat is sitting on the mat",
     "Stock prices rose sharply today",
     "非類似"),
    ("The weather is beautiful today",
     "It is a lovely sunny day",
     "類似"),
    ("The weather is beautiful today",
     "He plays guitar in a rock band",
     "非類似"),
    ("I love eating Italian food",
     "Pizza and pasta are my favorites",
     "類似"),
    ("I love eating Italian food",
     "The airplane landed safely at the airport",
     "非類似"),
]

# 各プーリング戦略で類似度を計算
pooling_strategies = {
    '[CLS] token': cls_pooling,
    'Mean Pooling': mean_pooling,
    'Max Pooling': max_pooling,
}

# 結果格納: (戦略数, ペア数)
results = {}
pair_labels = []
pair_types = []  # '類似' or '非類似'

for strategy_name, pooling_fn in pooling_strategies.items():
    sims = []
    for sent_a, sent_b, pair_type in test_pairs:
        emb_a = encode_with_bert([sent_a], bert_tokenizer, bert_model, pooling_fn)
        emb_b = encode_with_bert([sent_b], bert_tokenizer, bert_model, pooling_fn)
        sim = cosine_sim(emb_a[0], emb_b[0])
        sims.append(sim)
    results[strategy_name] = sims

# ペアラベルの作成（初回のみ）
for i, (sent_a, sent_b, pair_type) in enumerate(test_pairs):
    short_a = sent_a[:20] + "..."
    short_b = sent_b[:20] + "..."
    pair_labels.append(f"{pair_type}")
    pair_types.append(pair_type)

print("全ペアの類似度計算完了")
print()

# 結果表示
print(f"{'ペア':<8} {'[CLS]':>10} {'Mean':>10} {'Max':>10} {'期待'}")
print("-" * 55)
for i, (sent_a, sent_b, pair_type) in enumerate(test_pairs):
    cls_s = results['[CLS] token'][i]
    mean_s = results['Mean Pooling'][i]
    max_s = results['Max Pooling'][i]
    print(f"Pair {i+1:<3} {cls_s:>10.4f} {mean_s:>10.4f} {max_s:>10.4f}  {pair_type}")

In [ ]:
# ============================================================
# Plot 1: 3戦略の類似度スコア比較（棒グラフ）
# 類似ペアと非類似ペアをどれだけ分離できるか
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- 左: グループ化棒グラフ ---
x = np.arange(len(test_pairs))
width = 0.25
colors_strategy = ['#e74c3c', '#3498db', '#2ecc71']

for i, (strategy_name, sims) in enumerate(results.items()):
    bars = axes[0].bar(x + i * width, sims, width,
                       label=strategy_name, color=colors_strategy[i],
                       edgecolor='black', linewidth=0.5)

# 類似/非類似の背景色
for i, pair_type in enumerate(pair_types):
    if pair_type == '類似':
        axes[0].axvspan(i - 0.4, i + 0.8, alpha=0.08, color='green')
    else:
        axes[0].axvspan(i - 0.4, i + 0.8, alpha=0.08, color='red')

axes[0].set_xlabel('文ペア', fontsize=12)
axes[0].set_ylabel('コサイン類似度', fontsize=12)
axes[0].set_title('プーリング戦略ごとの文類似度スコア', fontsize=14)
axes[0].set_xticks(x + width)
axes[0].set_xticklabels([f'Pair {i+1}\n({t})' for i, t in enumerate(pair_types)],
                        fontsize=9)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

# --- 右: 分離度（類似平均 - 非類似平均）の比較 ---
strategy_names_list = list(results.keys())
separations = []
sim_means = []
dissim_means = []

for strategy_name in strategy_names_list:
    sims = results[strategy_name]
    # 類似ペアの平均（index 0, 2, 4）
    similar_avg = np.mean([sims[i] for i in range(len(sims)) if pair_types[i] == '類似'])
    # 非類似ペアの平均（index 1, 3, 5）
    dissimilar_avg = np.mean([sims[i] for i in range(len(sims)) if pair_types[i] == '非類似'])
    separations.append(similar_avg - dissimilar_avg)
    sim_means.append(similar_avg)
    dissim_means.append(dissimilar_avg)

x2 = np.arange(len(strategy_names_list))
width2 = 0.3

axes[1].bar(x2 - width2/2, sim_means, width2,
            label='類似ペア平均', color='#27ae60', edgecolor='black')
axes[1].bar(x2 + width2/2, dissim_means, width2,
            label='非類似ペア平均', color='#c0392b', edgecolor='black')

# 分離度をテキストで表示
for i, sep in enumerate(separations):
    axes[1].annotate(f'分離度: {sep:.3f}',
                     xy=(i, max(sim_means[i], dissim_means[i]) + 0.02),
                     ha='center', fontsize=10, fontweight='bold')

axes[1].set_xlabel('プーリング戦略', fontsize=12)
axes[1].set_ylabel('平均コサイン類似度', fontsize=12)
axes[1].set_title('類似/非類似ペアの分離度比較', fontsize=14)
axes[1].set_xticks(x2)
axes[1].set_xticklabels(strategy_names_list, fontsize=10)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim(0, 1.15)

plt.tight_layout()
plt.show()

print("観察ポイント:")
print("  分離度が大きいほど、類似/非類似を正しく区別できている")
print("  生のBERTでは、どの戦略も分離度が低い傾向がある")
print("  → Sentence-BERT（Section 4）で大幅に改善されます")

---

## 4. Sentence-Transformers の使い方

**sentence-transformers** ライブラリは、文埋め込みに特化したモデルを  
簡単に使えるPythonライブラリです。

### all-MiniLM-L6-v2

- 6層のTransformer（軽量）
- 384次元の出力
- 英語テキストに最適化
- Mean Pooling + 正規化 を内部で実行
- 速度と品質のバランスが良い

In [ ]:
# ============================================================
# Section 4: Sentence-Transformers による文エンコード
# model.encode() で簡単に文ベクトルを取得
# ============================================================

print("="*70)
print("Sentence-Transformers: 文埋め込みの生成")
print("="*70)
print()

# 意味的に類似/非類似な文のセット
sentences_demo = [
    # グループ1: 天気
    "The weather is lovely today",              # 0
    "It is a beautiful and sunny day",           # 1
    "Today has wonderful weather",               # 2
    # グループ2: プログラミング
    "Python is a great programming language",    # 3
    "I love coding in Python",                   # 4
    "Programming with Python is enjoyable",      # 5
    # グループ3: 料理
    "I enjoy cooking Italian dishes",            # 6
    "Making pasta from scratch is fun",          # 7
    "Italian cuisine is my favorite",            # 8
    # アウトライアー
    "The stock market crashed yesterday",        # 9
]

# Sentence-Transformerでエンコード
# normalize_embeddings=True: L2正規化（コサイン類似度 = ドット積）
embeddings_st = st_model.encode(
    sentences_demo,
    normalize_embeddings=True,
    show_progress_bar=False
)

print(f"入力文数: {len(sentences_demo)}")
print(f"埋め込み行列の形状: {embeddings_st.shape}")
print(f"各ベクトルの次元数: {embeddings_st.shape[1]}")
print(f"ベクトルのL2ノルム（正規化済み）: {np.linalg.norm(embeddings_st[0]):.4f}")
print()

# 特定ペアの類似度を確認
print("意味的に類似するペアの類似度:")
print(f"  天気1 vs 天気2: {cosine_sim(embeddings_st[0], embeddings_st[1]):.4f}")
print(f"  Python1 vs Python2: {cosine_sim(embeddings_st[3], embeddings_st[4]):.4f}")
print(f"  料理1 vs 料理2: {cosine_sim(embeddings_st[6], embeddings_st[7]):.4f}")
print()
print("意味的に非類似なペアの類似度:")
print(f"  天気1 vs Python1: {cosine_sim(embeddings_st[0], embeddings_st[3]):.4f}")
print(f"  料理1 vs 株式市場: {cosine_sim(embeddings_st[6], embeddings_st[9]):.4f}")
print(f"  天気1 vs 株式市場: {cosine_sim(embeddings_st[0], embeddings_st[9]):.4f}")

In [ ]:
# ============================================================
# Plot 2: 文ペア間の類似度ヒートマップ
# Sentence-Transformer による意味的クラスタリング
# ============================================================

# コサイン類似度行列を計算
# 正規化済みなので dot product = cosine similarity
sim_matrix = embeddings_st @ embeddings_st.T

# 短い表示用ラベル
short_labels = [
    "天気1", "天気2", "天気3",
    "Python1", "Python2", "Python3",
    "料理1", "料理2", "料理3",
    "株式市場"
]

fig, ax = plt.subplots(figsize=(10, 8))

# ヒートマップ描画
sns.heatmap(
    sim_matrix, ax=ax,
    xticklabels=short_labels,
    yticklabels=short_labels,
    annot=True, fmt='.2f',
    cmap='RdYlBu_r',
    vmin=-0.2, vmax=1.0,
    linewidths=0.5,
    annot_kws={'size': 9}
)

ax.set_title('Sentence-Transformer: 文ペア間のコサイン類似度', fontsize=14)

# グループ境界線を追加
for pos in [3, 6, 9]:
    ax.axhline(y=pos, color='black', linewidth=2)
    ax.axvline(x=pos, color='black', linewidth=2)

plt.tight_layout()
plt.show()

print("観察ポイント:")
print("  同じトピック内（対角ブロック）: 高い類似度（暖色）")
print("  異なるトピック間: 低い類似度（寒色）")
print("  株式市場の文は全てのグループと低い類似度を示す")
print("  → Sentence-Transformerは意味的なクラスタを正しく捉えている")

---

## 5. 意味検索（Semantic Search）の実装

文埋め込みの最も実用的な応用の1つが **意味検索（Semantic Search）** です。

### キーワード検索 vs 意味検索

| 特徴 | キーワード検索（TF-IDF等） | 意味検索（Embedding） |
|---|---|---|
| マッチング | 表層的な単語一致 | 意味的な類似性 |
| 同義語対応 | 不可（別の単語として扱う） | 可能（意味が近ければヒット） |
| 計算方法 | 単語頻度ベース | ベクトル類似度ベース |
| 例 | "dog" で検索 → "canine" はヒットしない | "dog" で検索 → "canine" もヒット |

In [ ]:
# ============================================================
# Section 5: 意味検索の実装
# コーパスに対するクエリ検索をTF-IDFと比較
# ============================================================

# コーパス: 多様なトピックの短い文（英語・日本語混合の想定）
corpus = [
    "The cat sat on the windowsill and watched the birds",           # 0
    "Machine learning models require large amounts of training data",# 1
    "A dog was playing fetch in the park with its owner",            # 2
    "Neural networks can learn complex patterns from data",          # 3
    "The kitten curled up on the sofa and fell asleep",              # 4
    "Deep learning has revolutionized computer vision",              # 5
    "Fresh pasta tastes much better than dried pasta",               # 6
    "The puppy chased its tail in circles around the yard",         # 7
    "Gradient descent is used to optimize model parameters",        # 8
    "Italian restaurants serve delicious pizza and pasta",           # 9
    "The parrot mimicked human speech perfectly",                    # 10
    "Transformers use self-attention for sequence modeling",         # 11
    "Cooking with fresh herbs enhances the flavor of food",         # 12
    "The goldfish swam in circles in its small bowl",               # 13
    "Backpropagation computes gradients for weight updates",        # 14
    "Sushi is a traditional Japanese delicacy",                      # 15
    "The bird built a nest high up in the oak tree",                # 16
    "Convolutional neural networks excel at image tasks",           # 17
]

print(f"コーパスサイズ: {len(corpus)} 文")
print()

# ============================================================
# コーパスをエンコード
# ============================================================

# Sentence-Transformer でエンコード
corpus_embeddings = st_model.encode(
    corpus,
    normalize_embeddings=True,
    show_progress_bar=False
)

# TF-IDF でエンコード
tfidf_vectorizer = TfidfVectorizer()
corpus_tfidf = tfidf_vectorizer.fit_transform(corpus)

print(f"Sentence-Transformer 埋め込み: {corpus_embeddings.shape}")
print(f"TF-IDF 埋め込み: {corpus_tfidf.shape}")
print()


def semantic_search(query, corpus_embs, corpus_texts, model, top_k=5):
    """
    Sentence-Transformer を使った意味検索。
    クエリをエンコードし、コーパスとのコサイン類似度で上位k件を返す。
    """
    # クエリをエンコード
    query_emb = model.encode([query], normalize_embeddings=True)
    # 類似度計算（正規化済みなのでドット積 = コサイン類似度）
    scores = (query_emb @ corpus_embs.T)[0]
    # 上位k件のインデックス
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(idx, scores[idx], corpus_texts[idx]) for idx in top_indices]


def tfidf_search(query, vectorizer, corpus_matrix, corpus_texts, top_k=5):
    """
    TF-IDF ベースのキーワード検索。
    """
    query_vec = vectorizer.transform([query])
    scores = sklearn_cosine(query_vec, corpus_matrix)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(idx, scores[idx], corpus_texts[idx]) for idx in top_indices]


print("検索関数の定義完了")

In [ ]:
# ============================================================
# 検索クエリで TF-IDF vs Semantic Search を比較
# 同義語やパラフレーズへの対応力をテスト
# ============================================================

# テストクエリ（意図的にコーパスと異なる表現を使用）
test_queries = [
    "cute animals resting at home",          # 動物が家で休む → cat/kitten
    "AI and artificial intelligence methods", # ML/DL関連 → machine learning系
    "delicious food from Japan",             # 日本食 → sushi
]

for query in test_queries:
    print("=" * 70)
    print(f"クエリ: \"{query}\"")
    print("=" * 70)
    
    # Semantic Search
    sem_results = semantic_search(
        query, corpus_embeddings, corpus, st_model, top_k=3
    )
    print("\n【Semantic Search（意味検索）】")
    for rank, (idx, score, text) in enumerate(sem_results, 1):
        print(f"  {rank}. [スコア: {score:.4f}] {text}")
    
    # TF-IDF Search
    tfidf_results = tfidf_search(
        query, tfidf_vectorizer, corpus_tfidf, corpus, top_k=3
    )
    print("\n【TF-IDF Search（キーワード検索）】")
    for rank, (idx, score, text) in enumerate(tfidf_results, 1):
        print(f"  {rank}. [スコア: {score:.4f}] {text}")
    print()

In [ ]:
# ============================================================
# Plot 3: 検索結果のランキング比較
# 各クエリでの上位結果のスコア分布を可視化
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for q_idx, query in enumerate(test_queries):
    ax = axes[q_idx]
    
    # Semantic Search の全文スコア
    query_emb = st_model.encode([query], normalize_embeddings=True)
    sem_scores = (query_emb @ corpus_embeddings.T)[0]
    
    # TF-IDF の全文スコア
    query_tfidf = tfidf_vectorizer.transform([query])
    tfidf_scores = sklearn_cosine(query_tfidf, corpus_tfidf)[0]
    
    # 上位5件のインデックス（Semantic）
    top5_sem = np.argsort(sem_scores)[::-1][:5]
    
    # 棒グラフ: Semantic vs TF-IDF
    x = np.arange(5)
    width = 0.35
    
    # Semanticの上位5件のスコア
    sem_top_scores = [sem_scores[i] for i in top5_sem]
    # 同じ文のTF-IDFスコア
    tfidf_top_scores = [tfidf_scores[i] for i in top5_sem]
    
    bars1 = ax.bar(x - width/2, sem_top_scores, width,
                   label='Semantic', color='#3498db', edgecolor='black')
    bars2 = ax.bar(x + width/2, tfidf_top_scores, width,
                   label='TF-IDF', color='#e67e22', edgecolor='black')
    
    # x軸ラベル: 文の短縮表示
    xlabels = [corpus[i][:25] + "..." for i in top5_sem]
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('類似度スコア', fontsize=10)
    ax.set_title(f'Query: "{query[:30]}..."', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 1.0)

fig.suptitle('Semantic Search vs TF-IDF: 検索結果の比較', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("観察ポイント:")
print("  Semantic Searchは同義語やパラフレーズに対応できる")
print("  TF-IDFは単語が一致しないとスコアが0になる")
print("  例: 'cute animals' → TF-IDFでは cat/kitten にヒットしにくい")

---

## 6. Sentence-BERTの学習の仕組み

### なぜBERTの[CLS]をそのまま使うより良いのか？

BERTの[CLS]トークンは **MLM（Masked Language Modeling）** と  
**NSP（Next Sentence Prediction）** のために学習されており、  
**文の意味的類似度** を捉えるようには最適化されていません。

Reimers & Gurevych (2019) は、BERTの[CLS]をそのまま文埋め込みに使うと、  
GloVe の平均ベクトルよりも **悪い** 性能になることを示しました。

### Siamese Network 構造

Sentence-BERT は **Siamese / Triplet Network** 構造で学習されます。

```
【Siamese BERT アーキテクチャ】

    文A: "A cat sits on a mat"       文B: "A kitten rests on a rug"
         │                                 │
         ▼                                 ▼
    ┌─────────┐                       ┌─────────┐
    │  BERT   │   重み共有            │  BERT   │
    │(同一モデル)│ ◄──────────────────► │(同一モデル)│
    └────┬────┘                       └────┬────┘
         │                                 │
         ▼                                 ▼
    ┌─────────┐                       ┌─────────┐
    │ Pooling │                       │ Pooling │
    │ (Mean)  │                       │ (Mean)  │
    └────┬────┘                       └────┬────┘
         │                                 │
         ▼                                 ▼
     ベクトル u                         ベクトル v
         │                                 │
         └──────────┬──────────────────────┘
                    │
                    ▼
             ┌─────────────┐
             │ 損失関数    │
             │ (後述)      │
             └─────────────┘
```

### 学習に使われるデータと損失関数

#### 1. NLI（Natural Language Inference）データ

- SNLI / MultiNLI データセット
- 各サンプル: (前提文, 仮説文, ラベル)
- ラベル: entailment（含意）/ contradiction（矛盾）/ neutral（中立）

**Softmax Loss:**

```
入力: concat(u, v, |u-v|)  →  Dense層  →  3クラスSoftmax
```

#### 2. STS（Semantic Textual Similarity）データ

- 文ペアに0-5の類似度スコアが付与

**Cosine Similarity Loss:**

```
Loss = MSE(cosine_sim(u, v), gold_score / 5.0)
```

In [ ]:
# ============================================================
# Section 6: Sentence-BERT の学習を簡易シミュレーション
# Siamese構造と損失関数の動作を理解する
# ============================================================

print("="*70)
print("Sentence-BERT の学習: 損失関数のシミュレーション")
print("="*70)
print()

# ============================================================
# Cosine Similarity Loss のデモ
# 人間が付けた類似度スコアに近づくよう学習する
# ============================================================

print("【Cosine Similarity Loss】")
print("-" * 50)
print("MSE(cosine_sim(u, v), target_score)")
print()

# シミュレーション用のペア
loss_demo_pairs = [
    ("A man is playing guitar", "A person plays a musical instrument", 4.2),
    ("A cat sleeps on the couch", "The stock market is booming", 0.3),
    ("Children are playing in the park", "Kids play outdoors", 4.5),
    ("A woman reads a book", "The dog chases a ball", 0.8),
]

print(f"{'文A':<35} {'文B':<35} {'目標':>5} {'予測':>5} {'Loss':>8}")
print("-" * 100)

total_loss = 0.0
for sent_a, sent_b, target_score in loss_demo_pairs:
    # Sentence-Transformerで埋め込みを取得
    emb_a = st_model.encode([sent_a], normalize_embeddings=True)[0]
    emb_b = st_model.encode([sent_b], normalize_embeddings=True)[0]
    
    # コサイン類似度（-1 ~ 1）を計算
    pred_sim = cosine_sim(emb_a, emb_b)
    # 目標スコアを 0-1 に正規化
    target_normalized = target_score / 5.0
    
    # MSE Loss
    loss = (pred_sim - target_normalized) ** 2
    total_loss += loss
    
    short_a = sent_a[:33] + (".." if len(sent_a) > 33 else "")
    short_b = sent_b[:33] + (".." if len(sent_b) > 33 else "")
    print(f"{short_a:<35} {short_b:<35} {target_normalized:>5.2f} {pred_sim:>5.2f} {loss:>8.4f}")

avg_loss = total_loss / len(loss_demo_pairs)
print("-" * 100)
print(f"{'平均 Loss:':<70} {' ':>5} {' ':>5} {avg_loss:>8.4f}")
print()
print("学習済みモデルのため、予測値と目標値の差（Loss）が小さい")
print("学習中はこのLossを最小化するよう重みが更新される")

---

## 7. 文埋め込みの品質評価

### STS（Semantic Textual Similarity）ベンチマーク

文埋め込みの品質は、**STS ベンチマーク** で評価するのが標準です。

- 人間が文ペアに 0（無関係）〜 5（同義）のスコアを付与
- モデルの予測スコアと人間のスコアの **Spearman 相関係数** で評価
- 相関が高いほど、モデルが人間の類似度判断に近い

### 評価指標: Spearman 相関係数

- 値の範囲: -1 〜 +1
- +1: 完全な正の単調関係
- 0: 相関なし
- -1: 完全な負の単調関係
- 順位ベースの相関なので、非線形な関係も捉えられる

In [ ]:
# ============================================================
# Section 7: 簡易STSタスクで品質評価
# 人間スコアとモデルスコアのSpearman相関を計算
# ============================================================

# 簡易STSデータセット（人間によるスコア付き）
# スコア: 0.0（完全に無関係）〜 5.0（同じ意味）
sts_data = [
    ("A man is playing a guitar",
     "Someone plays a musical instrument", 3.8),
    ("A cat is sitting on a mat",
     "A kitten rests on a rug", 4.2),
    ("The sun is shining brightly",
     "It is a beautiful sunny day", 4.0),
    ("A woman is reading a book",
     "A lady reads a novel", 4.5),
    ("Children are playing in the park",
     "Kids enjoy outdoor activities", 3.5),
    ("The dog chases a ball",
     "A puppy runs after a toy", 4.0),
    ("It is raining heavily outside",
     "There is a terrible storm", 3.2),
    ("A man is eating pizza",
     "Someone is having lunch", 3.0),
    ("The car is driving fast",
     "A plane flew over the mountain", 1.0),
    ("A person is riding a bicycle",
     "The stock market crashed today", 0.2),
    ("The teacher explains the lesson",
     "Fish swim in the ocean", 0.1),
    ("A boy kicks a soccer ball",
     "A child plays football", 4.3),
    ("The flowers bloom in spring",
     "Plants grow in warm weather", 3.0),
    ("A scientist conducts an experiment",
     "A researcher works in a laboratory", 3.8),
    ("The bird sings in the tree",
     "A computer processes data quickly", 0.0),
]

print(f"簡易STSデータセット: {len(sts_data)} ペア")
print()

# ============================================================
# 各手法でスコアを計算
# ============================================================

human_scores = [score for _, _, score in sts_data]

# 1. Sentence-Transformer スコア
st_scores = []
for sent_a, sent_b, _ in sts_data:
    emb_a = st_model.encode([sent_a], normalize_embeddings=True)[0]
    emb_b = st_model.encode([sent_b], normalize_embeddings=True)[0]
    st_scores.append(cosine_sim(emb_a, emb_b))

# 2. BERT [CLS] スコア
cls_scores = []
for sent_a, sent_b, _ in sts_data:
    emb_a = encode_with_bert([sent_a], bert_tokenizer, bert_model, cls_pooling)
    emb_b = encode_with_bert([sent_b], bert_tokenizer, bert_model, cls_pooling)
    cls_scores.append(cosine_sim(emb_a[0], emb_b[0]))

# 3. BERT Mean Pooling スコア
mean_scores = []
for sent_a, sent_b, _ in sts_data:
    emb_a = encode_with_bert([sent_a], bert_tokenizer, bert_model, mean_pooling)
    emb_b = encode_with_bert([sent_b], bert_tokenizer, bert_model, mean_pooling)
    mean_scores.append(cosine_sim(emb_a[0], emb_b[0]))

# 4. TF-IDF スコア
tfidf_sts_scores = []
for sent_a, sent_b, _ in sts_data:
    vec = TfidfVectorizer()
    tfidf_mat = vec.fit_transform([sent_a, sent_b])
    sim = sklearn_cosine(tfidf_mat[0:1], tfidf_mat[1:2])[0][0]
    tfidf_sts_scores.append(sim)

# ============================================================
# Spearman相関係数を計算
# ============================================================

methods = {
    'Sentence-Transformer': st_scores,
    'BERT [CLS]': cls_scores,
    'BERT Mean Pooling': mean_scores,
    'TF-IDF': tfidf_sts_scores,
}

print("="*60)
print("STS評価結果: Spearman相関係数")
print("="*60)
print()

spearman_results = {}
for method_name, scores in methods.items():
    corr, p_value = spearmanr(human_scores, scores)
    spearman_results[method_name] = corr
    print(f"  {method_name:<25}: r = {corr:.4f}  (p = {p_value:.6f})")

print()
print("Spearman相関が高いほど、人間の類似度判断に近い")

In [ ]:
# ============================================================
# Plot 4: 人間スコア vs モデルスコアの散布図
# 各手法がどれだけ人間の判断に近いかを可視化
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

method_items = list(methods.items())
colors_scatter = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']

for ax_idx, ((method_name, scores), color) in enumerate(zip(method_items, colors_scatter)):
    ax = axes[ax_idx // 2][ax_idx % 2]
    
    # 人間スコアを 0-1 に正規化（コサイン類似度と比較しやすくするため）
    human_normalized = [s / 5.0 for s in human_scores]
    
    # 散布図
    ax.scatter(human_normalized, scores, c=color,
               s=80, alpha=0.7, edgecolors='black', linewidths=0.5)
    
    # 理想線（y=x）
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='理想 (y=x)')
    
    # 回帰線
    z = np.polyfit(human_normalized, scores, 1)
    p = np.poly1d(z)
    x_line = np.linspace(0, 1, 100)
    ax.plot(x_line, p(x_line), color=color, linewidth=2, alpha=0.7,
            label=f'回帰線')
    
    # Spearman相関をテキスト表示
    corr = spearman_results[method_name]
    ax.text(0.05, 0.95, f'Spearman r = {corr:.4f}',
            transform=ax.transAxes, fontsize=12, fontweight='bold',
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.set_xlabel('人間スコア（正規化: 0-1）', fontsize=11)
    ax.set_ylabel('モデルスコア（コサイン類似度）', fontsize=11)
    ax.set_title(method_name, fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.1, 1.1)

fig.suptitle('STS評価: 人間スコア vs モデルスコア', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

print("観察ポイント:")
print("  Sentence-Transformer: 点が対角線に近い → 人間の判断に最も近い")
print("  BERT [CLS]: 全体的に高いスコアに偏り、分離が悪い")
print("  BERT Mean: [CLS]よりは良いが、Sentence-Transformerには及ばない")
print("  TF-IDF: 単語一致に依存するため、パラフレーズに弱い")

In [ ]:
# ============================================================
# Plot 5: Spearman相関係数の棒グラフ比較
# 各手法のSTS性能を一目で比較
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6))

method_names = list(spearman_results.keys())
correlations = list(spearman_results.values())
bar_colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']

bars = ax.bar(method_names, correlations, color=bar_colors,
              edgecolor='black', linewidth=1.2, width=0.6)

# 値ラベル
for bar, corr in zip(bars, correlations):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{corr:.4f}', ha='center', va='bottom',
            fontsize=13, fontweight='bold')

ax.set_ylabel('Spearman 相関係数', fontsize=12)
ax.set_title('STS評価: 各手法のSpearman相関係数', fontsize=14)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1.1)

# 参考線
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5)
ax.text(3.5, 0.81, '良好な基準 (0.8)', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

print("結論:")
print("  Sentence-Transformerは文の意味的類似度を最も正確に捉える")
print("  BERTの[CLS]をそのまま使うのは推奨されない")
print("  Mean Poolingは手軽な改善策だが、専用モデルには及ばない")

---

## 8. 文書レベルへの拡張

Sentence-BERT は短い文（通常 128〜512 トークン）向けに設計されています。  
長い文書を埋め込むには追加の工夫が必要です。

### 長い文書の埋め込み手法

| 手法 | 説明 | 長所 | 短所 |
|---|---|---|---|
| **チャンク分割 + 平均** | 文書を短い断片に分割し、各断片の埋め込みを平均 | シンプル、追加学習不要 | 文書構造の情報が失われる |
| **階層的アプローチ** | 文→段落→文書の階層的な集約 | 構造を保持 | 実装が複雑 |
| **専用モデル** | Longformer, BigBird等の長文対応モデル | 長い文脈を直接処理 | 計算コストが高い |

### トークン長制限への対処法

```
【チャンク分割 + 平均】

長い文書: "This is a very long document... (2000 tokens)"
    │
    ├─ チャンク1 (tokens 0-255)   → 埋め込み e1
    ├─ チャンク2 (tokens 256-511) → 埋め込み e2
    ├─ チャンク3 (tokens 512-767) → 埋め込み e3
    └─ ...
    │
    ▼
    文書埋め込み = mean(e1, e2, e3, ...)

【オーバーラップ付きチャンク分割】

    チャンク1: [tokens 0 ---- 255]
    チャンク2:     [tokens 64 ---- 319]  ← 64トークンのオーバーラップ
    チャンク3:         [tokens 128 ---- 383]
    → 文脈の断絶を軽減
```

### 実用上の注意点

1. **チャンクサイズの選択**: モデルの最大入力長の80%程度が目安
2. **オーバーラップ**: 10-25%のオーバーラップで文脈断絶を軽減
3. **重み付き平均**: 文書の先頭部分に高い重みを付けることも有効
4. **正規化**: 最終的な文書ベクトルはL2正規化すること

In [ ]:
# ============================================================
# Section 8: 文書レベルの埋め込み — チャンク分割戦略の実装
# 長い文書を短いチャンクに分割してエンコードする
# ============================================================

def chunk_text(text, chunk_size=3, overlap=1):
    """
    テキストを文単位でチャンクに分割する。
    
    Parameters:
    -----------
    text : str
        入力テキスト（ピリオドで文分割）
    chunk_size : int
        1チャンクあたりの文数
    overlap : int
        チャンク間のオーバーラップ文数
    
    Returns:
    --------
    list of str : チャンクのリスト
    """
    # 簡易的な文分割（ピリオド区切り）
    sentences = [s.strip() for s in text.split('.') if s.strip()]
    
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(sentences), max(step, 1)):
        chunk = '. '.join(sentences[i:i + chunk_size]) + '.'
        chunks.append(chunk)
        if i + chunk_size >= len(sentences):
            break
    
    return chunks


def encode_long_document(text, model, chunk_size=3, overlap=1):
    """
    長い文書をチャンク分割 + 平均でエンコードする。
    
    Returns:
    --------
    np.ndarray : 文書の埋め込みベクトル
    list of str : チャンクのリスト
    """
    # チャンク分割
    chunks = chunk_text(text, chunk_size, overlap)
    
    # 各チャンクをエンコード
    chunk_embeddings = model.encode(
        chunks,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    
    # 平均してL2正規化
    doc_embedding = chunk_embeddings.mean(axis=0)
    doc_embedding = doc_embedding / np.linalg.norm(doc_embedding)
    
    return doc_embedding, chunks


# ============================================================
# デモ: 2つのテーマの文書を比較
# ============================================================

# 機械学習に関する文書
doc_ml = (
    "Machine learning is a subset of artificial intelligence. "
    "It enables computers to learn from data without being explicitly programmed. "
    "Neural networks are a popular machine learning technique. "
    "Deep learning uses multiple layers to extract features. "
    "Training requires large datasets and significant compute resources. "
    "Common applications include image recognition and natural language processing."
)

# 料理に関する文書
doc_cooking = (
    "Italian cooking is known for its simplicity and fresh ingredients. "
    "Pasta is a staple food in Italian cuisine. "
    "Olive oil is used extensively in Mediterranean cooking. "
    "Tomatoes are a key ingredient in many Italian dishes. "
    "Fresh basil and oregano add wonderful flavor. "
    "Pizza originated in Naples and became popular worldwide."
)

# クエリ
query_ml = "How do AI systems learn from examples?"
query_cooking = "What ingredients are used in Mediterranean food?"

# 文書をエンコード
emb_ml, chunks_ml = encode_long_document(doc_ml, st_model)
emb_cooking, chunks_cooking = encode_long_document(doc_cooking, st_model)

# クエリをエンコード
emb_q_ml = st_model.encode([query_ml], normalize_embeddings=True)[0]
emb_q_cooking = st_model.encode([query_cooking], normalize_embeddings=True)[0]

print("="*70)
print("文書レベルの埋め込み: チャンク分割 + 平均")
print("="*70)
print()

# チャンクの確認
print(f"ML文書のチャンク数: {len(chunks_ml)}")
for i, chunk in enumerate(chunks_ml):
    print(f"  チャンク{i+1}: {chunk[:60]}...")
print()

# クエリと文書の類似度
print("クエリと文書の類似度:")
print(f"  ML質問 vs ML文書:    {cosine_sim(emb_q_ml, emb_ml):.4f}")
print(f"  ML質問 vs 料理文書:  {cosine_sim(emb_q_ml, emb_cooking):.4f}")
print(f"  料理質問 vs ML文書:  {cosine_sim(emb_q_cooking, emb_ml):.4f}")
print(f"  料理質問 vs 料理文書: {cosine_sim(emb_q_cooking, emb_cooking):.4f}")
print()
print("関連するクエリと文書のペアで高い類似度が得られている")

In [ ]:
# ============================================================
# Plot 6: 文書埋め込みのPCA可視化
# チャンク埋め込みと文書全体の埋め込みの関係
# ============================================================

# 全チャンクの埋め込みを取得
all_chunk_embs_ml = st_model.encode(chunks_ml, normalize_embeddings=True,
                                     show_progress_bar=False)
all_chunk_embs_cooking = st_model.encode(chunks_cooking, normalize_embeddings=True,
                                          show_progress_bar=False)

# PCAで2次元に射影
all_vecs = np.vstack([
    all_chunk_embs_ml,
    all_chunk_embs_cooking,
    emb_ml.reshape(1, -1),
    emb_cooking.reshape(1, -1),
    emb_q_ml.reshape(1, -1),
    emb_q_cooking.reshape(1, -1),
])

pca = PCA(n_components=2)
vecs_2d = pca.fit_transform(all_vecs)

fig, ax = plt.subplots(figsize=(12, 8))

n_ml = len(chunks_ml)
n_cooking = len(chunks_cooking)
offset = 0

# MLチャンク
ax.scatter(vecs_2d[offset:offset+n_ml, 0], vecs_2d[offset:offset+n_ml, 1],
           c='#3498db', marker='o', s=100, alpha=0.6, label='MLチャンク',
           edgecolors='black', linewidths=0.5)
offset += n_ml

# 料理チャンク
ax.scatter(vecs_2d[offset:offset+n_cooking, 0], vecs_2d[offset:offset+n_cooking, 1],
           c='#e67e22', marker='o', s=100, alpha=0.6, label='料理チャンク',
           edgecolors='black', linewidths=0.5)
offset += n_cooking

# ML文書全体
ax.scatter(vecs_2d[offset, 0], vecs_2d[offset, 1],
           c='#2c3e50', marker='*', s=400, label='ML文書(平均)',
           edgecolors='black', linewidths=1, zorder=10)
offset += 1

# 料理文書全体
ax.scatter(vecs_2d[offset, 0], vecs_2d[offset, 1],
           c='#d35400', marker='*', s=400, label='料理文書(平均)',
           edgecolors='black', linewidths=1, zorder=10)
offset += 1

# MLクエリ
ax.scatter(vecs_2d[offset, 0], vecs_2d[offset, 1],
           c='#3498db', marker='D', s=200, label='MLクエリ',
           edgecolors='red', linewidths=2, zorder=10)
offset += 1

# 料理クエリ
ax.scatter(vecs_2d[offset, 0], vecs_2d[offset, 1],
           c='#e67e22', marker='D', s=200, label='料理クエリ',
           edgecolors='red', linewidths=2, zorder=10)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=12)
ax.set_title('PCA可視化: 文書チャンクとクエリの埋め込み空間', fontsize=14)
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("観察ポイント:")
print("  同じ文書のチャンクは近い位置にクラスタリングされる")
print("  文書全体の埋め込み（星印）はチャンクの重心付近に位置")
print("  クエリ（ダイヤモンド）は関連する文書クラスタに近い")

---

## 9. まとめ・チートシート・よくある間違い・自己評価クイズ

In [ ]:
# ============================================================
# Section 9: まとめ
# 手法比較表と実践的ガイドライン
# ============================================================

print("="*70)
print("まとめ: 文・文書埋め込み手法の比較")
print("="*70)
print()
print(f"{'手法':<25} {'特徴':<30} {'計算コスト':<12} {'品質'}")
print("-" * 80)
print(f"{'BERT [CLS]':<25} {'最もシンプル':<30} {'中':<12} {'低'}")
print(f"{'BERT Mean Pooling':<25} {'手軽な改善':<30} {'中':<12} {'中'}")
print(f"{'BERT Max Pooling':<25} {'顕著な特徴を強調':<30} {'中':<12} {'中'}")
print(f"{'Sentence-BERT':<25} {'文類似度に特化':<30} {'中':<12} {'高'}")
print(f"{'all-MiniLM-L6-v2':<25} {'軽量・高速':<30} {'低':<12} {'高'}")
print(f"{'TF-IDF':<25} {'単語頻度ベース':<30} {'低':<12} {'低'}")
print(f"{'Cross-Encoder':<25} {'最高精度だが遅い':<30} {'非常に高':<12} {'最高'}")
print()

# ============================================================
# 実践的ガイドライン
# ============================================================
print("="*70)
print("実践ガイドライン")
print("="*70)
print()
print("【推奨ワークフロー】")
print("  1. まず Sentence-Transformer（all-MiniLM-L6-v2）を試す")
print("  2. 精度不足なら大きいモデル（all-mpnet-base-v2 等）に切替")
print("  3. ドメイン特化が必要なら、自前データでファインチューニング")
print("  4. 最高精度が必要なら、Bi-Encoder + Cross-Encoder のリランキング")
print()
print("【モデル選択の目安】")
print("  速度重視:    all-MiniLM-L6-v2（6層, 384次元）")
print("  バランス:    all-mpnet-base-v2（12層, 768次元）")
print("  多言語対応:  paraphrase-multilingual-MiniLM-L12-v2")
print("  日本語特化:  multilingual-e5-large 等")

In [ ]:
# ============================================================
# よくある間違い
# ============================================================

print("="*70)
print("よくある間違い")
print("="*70)
print()

print("【間違い1】BERTの[CLS]をそのまま文埋め込みに使う")
print("-" * 50)
print("  x: embeddings = bert_output.last_hidden_state[:, 0, :]")
print("  o: Sentence-Transformer を使用する")
print("  理由: BERTの[CLS]はMLM/NSP用に学習されており、")
print("        文の意味的類似度を正しく反映しない")
print("        GloVe平均よりも悪い結果になることがある")
print()

print("【間違い2】attention_maskを無視したMean Pooling")
print("-" * 50)
print("  x: embedding = outputs.mean(dim=1)  # パディングも含む")
print("  o: mask考慮の加重平均")
print("  理由: パディングトークンの埋め込み（ゼロに近い値）が")
print("        平均に混入し、ベクトルの品質が低下する")
print("        特にバッチ処理で文の長さが異なる場合に深刻")
print()

print("【間違い3】正規化せずにドット積で類似度計算")
print("-" * 50)
print("  x: similarity = np.dot(emb_a, emb_b)  # ベクトルの大きさに依存")
print("  o: similarity = cosine_similarity(emb_a, emb_b)  # 正規化済み")
print("  理由: ドット積はベクトルの大きさ（ノルム）に影響される")
print("        文の長さや表現の強さが結果を歪める")
print("        コサイン類似度を使うか、事前にL2正規化すること")

### 自己評価クイズ

以下の問題で理解度をチェックしましょう。

---

**Q1**: Cross-Encoder と Bi-Encoder の最大の違いは何ですか？  
10,000文の全ペア類似度を計算する場合、それぞれ何回のBERT推論が必要ですか？

<details>
<summary>回答を見る</summary>

**Cross-Encoder**: 2つの文を同時にBERTに入力して類似度を計算します。  
全ペア = 10,000 x 9,999 / 2 = **約5,000万回** の推論が必要。

**Bi-Encoder**: 各文を独立にエンコードし、ベクトル間の類似度を計算します。  
エンコード = **10,000回** の推論 + ベクトル比較は行列演算で瞬時。

Bi-Encoderは約5,000倍高速ですが、Cross-Encoderの方が精度は高い傾向があります。
</details>

---

**Q2**: Mean Pooling で `attention_mask` を考慮する必要がある理由を説明してください。

<details>
<summary>回答を見る</summary>

バッチ処理では、異なる長さの文を同じテンソルに格納するために **パディング** が追加されます。  
パディングトークンの埋め込みはゼロに近い値ですが、完全にゼロではありません。  

`attention_mask` を使わずに平均すると、これらのパディング埋め込みが混入し、  
短い文ほどパディングの影響が大きくなって **ベクトルの品質が低下** します。

`attention_mask` でパディング位置を除外して加重平均を取ることで、  
実際のトークンのみの正確な平均が得られます。
</details>

---

**Q3**: Sentence-BERT が通常のBERTの[CLS]よりも文類似度タスクで優れている理由は何ですか？

<details>
<summary>回答を見る</summary>

通常のBERTは **MLM（Masked Language Modeling）** と **NSP（Next Sentence Prediction）** で  
事前学習されており、文の **意味的類似度** を直接最適化していません。

Sentence-BERT は **Siamese Network** 構造で、NLI や STS データを使って  
**文ペアの意味的関係** を直接学習します。具体的には:

- NLIデータ: 含意/矛盾/中立 の3分類（Softmax Loss）
- STSデータ: コサイン類似度と人間スコアの回帰（Cosine Similarity Loss）

この追加学習により、文の意味を正確に反映した埋め込みが得られます。
</details>

---

**Q4**: 長い文書（例: 5000トークン）をSentence-Transformerでエンコードする場合、  
どのような方法が使えますか？それぞれの長所・短所を述べてください。

<details>
<summary>回答を見る</summary>

主な方法:

1. **チャンク分割 + 平均**: 文書を短い断片に分割し、各断片の埋め込みを平均。  
   長所: シンプル、追加学習不要。短所: 文書構造や長距離の文脈情報が失われる。

2. **階層的アプローチ**: 文→段落→文書の段階的な集約。  
   長所: 構造を保持。短所: 実装が複雑。

3. **専用モデル**: Longformer, BigBird 等の長文対応Transformer。  
   長所: 長い文脈を直接処理可能。短所: 計算コストが高い。

実用上は、オーバーラップ付きのチャンク分割 + 平均が最も手軽で効果的です。
</details>

In [ ]:
# ============================================================
# 次のステップ
# ============================================================

print("="*70)
print("Notebook 153 完了!")
print("="*70)
print()
print("【学んだこと】")
print("  1. 文埋め込みが必要な理由（Cross-Encoder vs Bi-Encoder）")
print("  2. プーリング戦略（CLS/Mean/Max）の実装と比較")
print("  3. Sentence-Transformersライブラリの使い方")
print("  4. 意味検索の実装とTF-IDFとの比較")
print("  5. Sentence-BERTの学習（Siamese Network + NLI/STS）")
print("  6. STSベンチマークによる品質評価（Spearman相関）")
print("  7. 長い文書のチャンク分割戦略")
print()
print("【次のステップ】")
print("  → Notebook 154: 多様体学習と可視化")
print("    t-SNE, UMAP による高次元埋め込みの可視化、")
print("    埋め込み空間の構造分析を学びます。")

---

## 参考文献

### 基本文献
1. Reimers, N. & Gurevych, I. (2019). Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks. *EMNLP*.
2. Devlin, J., et al. (2019). BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding. *NAACL-HLT*.

### 文埋め込み・検索
3. Reimers, N. & Gurevych, I. (2020). Making Monolingual Sentence Embeddings Multilingual using Knowledge Distillation. *EMNLP*.
4. Thakur, N., et al. (2021). BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models. *NeurIPS*.
5. Muennighoff, N., et al. (2023). MTEB: Massive Text Embedding Benchmark. *EACL*.

### 実践ガイド
6. Sentence-Transformers Documentation: https://www.sbert.net/
7. Hugging Face Hub (models): https://huggingface.co/models?library=sentence-transformers